In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd



"""
2 additional graphs were added to success and collision rate analysis. 
1. Collision intensity: Ratio of total collisions to total steps across all evaluation episodes. 
                        This provides a per-step collision frequency that is indifferent to episode length.
2. Episode breakdown: Failed episodes are classified into four mutually exclusive severity categories based on their collision count.
"""

agents = ['Random', 'DDPG', 'TD3', 'SAC', 'PPO', 'TRPO']
raw = {}
for a in agents:
    with open(rf'C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp\results_{a}.pkl', 'rb') as f:
        raw[a] = pickle.load(f)


breakdown = {}
for a in agents:
    colls = np.array(raw[a]['all_collision_counts'][a])
    succ  = np.array(raw[a]['all_successes'][a])
    steps = np.array(raw[a]['all_steps'][a])
    fail  = colls[succ == 0]
    breakdown[a] = {
        'success':      int((succ == 1).sum()),
        'wandering':    int((fail == 0).sum()),
        'low':          int(((fail > 0)   & (fail <= 100)).sum()),
        'moderate':     int(((fail > 100) & (fail <= 1000)).sum()),
        'catastrophic': int((fail > 1000).sum()),
        'intensity':    float(colls.sum() / steps.sum()),
        'n': 50,
    }


C_SUCCESS      = '#2E7D32'
C_WANDERING    = '#81C784'
C_LOW          = '#FFB300'
C_MODERATE     = '#E65100'
C_CATASTROPHIC = '#B71C1C'

x = np.arange(len(agents))
w = 0.55
n = 50

s  = np.array([breakdown[a]['success']      for a in agents])
wn = np.array([breakdown[a]['wandering']    for a in agents])
lo = np.array([breakdown[a]['low']          for a in agents])
mo = np.array([breakdown[a]['moderate']     for a in agents])
ca = np.array([breakdown[a]['catastrophic'] for a in agents])


# Episode-wise breakdown of collision added after success and collision rates alone were not enough. 

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x, wn,              width=w, color=C_WANDERING,    label='Wandering (0 collisions)',         zorder=3)
ax.bar(x, lo, bottom=wn,   width=w, color=C_LOW,          label='Low-collision failure (1–100)',     zorder=3)
ax.bar(x, mo, bottom=wn+lo,         width=w, color=C_MODERATE,     label='Moderate failure (101–1000)',      zorder=3)
ax.bar(x, ca, bottom=wn+lo+mo,      width=w, color=C_CATASTROPHIC, label='Catastrophic failure (>1000)',     zorder=3)

for i, a in enumerate(agents):
    c = breakdown[a]['catastrophic']
    if c > 0:
        top = wn[i] + lo[i] + mo[i] + ca[i]
        ax.text(i, top + 0.3, str(c), ha='center', va='bottom',
                fontsize=9, fontweight='bold', color=C_CATASTROPHIC)

ax.set_xticks(x)
ax.set_xticklabels(agents, fontsize=11)
ax.set_ylabel('Number of failed episodes', fontsize=11)
ax.set_ylim(0, 55)
tick_vals = [0, 10, 20, 30, 40, 50]
ax.yaxis.set_major_locator(FixedLocator(tick_vals))
ax.yaxis.set_major_formatter(FixedFormatter([str(v) for v in tick_vals]))
ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.6, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13),
          ncol=2, fontsize=9, frameon=False)
ax.set_title('Collision severity breakdown across failed episodes', fontsize=13, pad=10)

fig.tight_layout()
fig.savefig(rf'C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp\fig_episode_breakdown.pdf', bbox_inches='tight', dpi=300)
fig.savefig(rf'C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp\fig_episode_breakdown.png', bbox_inches='tight', dpi=300)
plt.close(fig)



intensities = [breakdown[a]['intensity'] for a in agents]
colors = [C_CATASTROPHIC if v > 0.5 else C_MODERATE if v > 0.2 else C_LOW
          for v in intensities]

fig2, ax2 = plt.subplots(figsize=(9, 4))
bars = ax2.bar(x, intensities, width=w, color=colors,
               zorder=3, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, intensities):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.012,
             f'{val:.3f}', ha='center', va='bottom',
             fontsize=9, fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels(agents, fontsize=11)
ax2.set_ylabel('Mean collisions per step', fontsize=11)
ax2.set_ylim(0, 1.05)
ax2.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.6, zorder=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.axhline(0.5, color='#555', linestyle=':', linewidth=1.2, zorder=2)
ax2.text(5.3, 0.515, 'high-risk threshold', fontsize=8,
         color='#555', va='bottom', ha='center')
ax2.set_title('Collision intensity per agent (mean collisions per step)',
              fontsize=13, pad=10)

fig2.tight_layout()

fig2.savefig(rf'C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp\fig_collision_intensity.pdf', bbox_inches='tight', dpi=300)
fig2.savefig(rf'C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp\fig_collision_intensity.png', bbox_inches='tight', dpi=300)
plt.close(fig2)

## Summary Metrics Table

Computes and prints a per-agent summary metrics table across all evaluation episodes.
Results are loaded from the per-agent `.pkl` files produced during training.

Each row is one metric; each column is one agent. Metrics cover:
- **Reward**: mean, std, median, max, min, and breakdowns by success/failure
- **Steps**: mean, std, median, and mean on successful episodes
- **Success, collision, and timeout rates**
- **Collision severity**: mean count, intensity (collisions per step), catastrophic episodes (>1000 collisions)
- **Risk-adjusted score**: success rate penalised by catastrophic failure rate

These results correspond to Table 6 in Appendix 6 of the thesis.

In [ ]:

agents = ['Random', 'DDPG', 'TD3', 'SAC', 'PPO', 'TRPO']
raw = {}
for a in agents:
    with open(rf'C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp\results_{a}.pkl', 'rb') as f:
        raw[a] = pickle.load(f)

print("Metric".ljust(35), " | ".join(agents))
print("-" * 90)

for a in agents:
    d = raw[a]
    rewards = np.array(d['all_rewards'][a])
    steps   = np.array(d['all_steps'][a])
    colls   = np.array(d['all_collision_counts'][a])
    succ    = np.array(d['all_successes'][a])
    coll_b  = np.array(d['all_collisions'][a])
    tout    = np.array(d['all_timeouts'][a])

    raw[a]['_r'] = rewards
    raw[a]['_s'] = steps
    raw[a]['_c'] = colls
    raw[a]['_succ'] = succ
    raw[a]['_cb'] = coll_b
    raw[a]['_t'] = tout

def row(label, vals):
    print(f"{label:<35}", "  ".join(f"{v:>10}" for v in vals))

def fmt(v, dec=2):
    return f"{v:.{dec}f}"

metrics = {}
for a in agents:
    r = raw[a]['_r']
    s = raw[a]['_s']
    c = raw[a]['_c']
    succ = raw[a]['_succ']
    cb = raw[a]['_cb']
    t = raw[a]['_t']

    succ_r = r[succ==1]
    fail_r = r[succ==0]
    succ_s = s[succ==1]
    succ_c = c[succ==1]
    fail_c = c[succ==0]

    metrics[a] = {
        'Mean Reward':              fmt(r.mean()),
        'Reward Std':               fmt(r.std()),
        'Median Reward':            fmt(np.median(r)),
        'Max Reward':               fmt(r.max()),
        'Min Reward':               fmt(r.min()),
        'Mean Reward (success)':    fmt(succ_r.mean()) if len(succ_r)>0 else '---', # guard against empty slices when an agent never succeeds or never fails
        'Mean Reward (fail)':       fmt(fail_r.mean()) if len(fail_r)>0 else '---',
        'Mean Steps':               fmt(s.mean()),
        'Steps Std':                fmt(s.std()),
        'Median Steps':             fmt(np.median(s)),
        'Mean Steps (success)':     fmt(succ_s.mean()) if len(succ_s)>0 else '---',
        'Success Rate':             fmt(succ.mean()),
        'Collision Rate':           fmt(cb.mean()),
        'Timeout Rate':             fmt(t.mean()),
        'Mean Collision Count':     fmt(c.mean()),
        'Collision Std':            fmt(c.std()),
        'Mean Coll (success)':      fmt(succ_c.mean()) if len(succ_c)>0 else '---',
        'Mean Coll (fail)':         fmt(fail_c.mean()) if len(fail_c)>0 else '---',
        'Collision Intensity':      fmt(c.sum()/s.sum(), 4),
        'Catastrophic (>1000)':     str(int((c>1000).sum())),
        'Wandering (=0, fail)':     str(int((fail_c==0).sum())) if len(fail_c)>0 else '---',
        'Risk-Adjusted Score':      fmt(succ.mean() - (c>1000).mean()),
    }

for metric in list(metrics[agents[0]].keys()):
    vals = [metrics[a][metric] for a in agents]
    row(metric, vals)

Metric                              Random | DDPG | TD3 | SAC | PPO | TRPO
------------------------------------------------------------------------------------------
Mean Reward                           -1185.90     -187.60     -101.12     -359.64     -530.11      -63.27
Reward Std                              346.14      589.22      599.81      749.59      796.31      295.05
Median Reward                         -1300.73       32.37      173.63      186.87      158.40       -7.36
Max Reward                             -158.33      265.83      387.96      387.49      387.37       54.67
Min Reward                            -1984.84    -1507.50    -1507.50    -1507.50    -1509.38    -1507.50
Mean Reward (success)                      ---      217.84      233.23      241.71      231.72         ---
Mean Reward (fail)                    -1185.90     -361.35     -526.66    -1124.99    -1355.43      -63.27
Mean Steps                             1500.00     1071.26      722.48      723.76   

In [ ]:

"""
Plots were regenerated because 1. DDPG and TD3's colours in Fig 1 were too similar and it was hard to read and 
2. because the colours in the subsequent graphs did not match anymore 
Alongside PDF, PNG are also created to be able to insert the images in Overleaf. 
"""


result_directory = r"C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp"
agents = ["Random", "DDPG", "TD3", "SAC", "PPO", "TRPO"]


color_palette = {
    "Random": "#757575",   
    "DDPG":   "#1B5E20",   
    "TD3":    "#0077BB",   
    "SAC":    "#CC3311",   
    "PPO":    "#EE7733",   
    "TRPO":   "#AA3377",   
}

def agent_color(name):
    return color_palette.get(name, "#607D8B")


raw = {}
for a in agents:
    path = os.path.join(result_directory, f"results_{a}.pkl")
    with open(path, "rb") as f:
        raw[a] = pickle.load(f)

training_curves      = {}
all_rewards          = {}
all_steps            = {}
all_collisions       = {}
all_successes        = {}

for a in agents:
    r = raw[a]
    if "training_curves" in r and a in r["training_curves"]:
        training_curves[a] = r["training_curves"][a]
    all_rewards[a]    = r["all_rewards"][a]
    all_steps[a]      = r["all_steps"][a]
    all_collisions[a] = r["all_collisions"][a]
    all_successes[a]  = r["all_successes"][a]


sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 150})

rl_names    = [n for n in agents if n in training_curves]
agent_names = agents  # keeps a fixed display order



if rl_names:
    fig1, ax1 = plt.subplots(figsize=(7, 4))
    for name in rl_names:
        xs = np.array(training_curves[name]["steps"])
        ys = np.array(training_curves[name]["rewards"])
        ax1.plot(xs, ys, label=name, color=agent_color(name),
                 linewidth=2, marker="o", markersize=4)
        ax1.fill_between(xs, ys * 0.95, ys * 1.05,
                         color=agent_color(name), alpha=0.15)
    ax1.set_xlabel("Environment steps")
    ax1.set_ylabel("Mean evaluation return")
    ax1.set_title("Training learning curves")
    ax1.legend(frameon=False)
    ax1.ticklabel_format(axis="x", style="sci", scilimits=(4, 4))
    fig1.tight_layout()
    fig1.savefig(os.path.join(result_directory, "fig1_learning_curves.pdf"), bbox_inches="tight")
    fig1.savefig(os.path.join(result_directory, "fig1_learning_curves.png"), bbox_inches="tight", dpi=300)
    plt.close(fig1)


reward_df = pd.DataFrame(
    {n: pd.Series(v) for n, v in all_rewards.items()}
).melt(var_name="Agent", value_name="Return")

fig2, ax2 = plt.subplots(figsize=(8, 4))
sns.boxplot(data=reward_df, x="Agent", y="Return",
            palette={n: agent_color(n) for n in agent_names},
            order=agent_names, width=0.5, linewidth=1.2,
            flierprops=dict(marker="x", markersize=4, linewidth=0.8), ax=ax2)
sns.stripplot(data=reward_df, x="Agent", y="Return",
              palette={n: agent_color(n) for n in agent_names},
              order=agent_names, dodge=False, jitter=True,
              size=3, alpha=0.5, ax=ax2)
ax2.set_title("Final evaluation return distribution")
ax2.set_xlabel("")
ax2.set_ylabel("Cumulative return")
ax2.tick_params(axis="x", rotation=15)
fig2.tight_layout()
fig2.savefig(os.path.join(result_directory, "fig2_final_return_boxplot.pdf"), bbox_inches="tight")
fig2.savefig(os.path.join(result_directory, "fig2_final_return_boxplot.png"), bbox_inches="tight", dpi=300)
plt.close(fig2)


steps_df = pd.DataFrame(
    {n: pd.Series(v) for n, v in all_steps.items()}
).melt(var_name="Agent", value_name="Steps")

fig3, ax3 = plt.subplots(figsize=(8, 4))
sns.violinplot(data=steps_df, x="Agent", y="Steps",
               palette={n: agent_color(n) for n in agent_names},
               order=agent_names, inner="quartile",
               linewidth=1.0, cut=0, ax=ax3)
ax3.set_title("Steps per episode")
ax3.set_xlabel("")
ax3.set_ylabel("Steps to termination")
ax3.tick_params(axis="x", rotation=15)
fig3.tight_layout()
fig3.savefig(os.path.join(result_directory, "fig3_steps_violin.pdf"), bbox_inches="tight")
fig3.savefig(os.path.join(result_directory, "fig3_steps_violin.png"), bbox_inches="tight", dpi=300)
plt.close(fig3)

print("All figures saved to", result_directory)

C:\Users\pirat\AppData\Local\Temp\ipykernel_67552\2470794087.py:88: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=reward_df, x="Agent", y="Return",
C:\Users\pirat\AppData\Local\Temp\ipykernel_67552\2470794087.py:92: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(data=reward_df, x="Agent", y="Return",
C:\Users\pirat\AppData\Local\Temp\ipykernel_67552\2470794087.py:111: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=steps_df, x="Agent", y="Steps",


All figures saved to C:\Users\pirat\BSc-Thesis-CSAI-Mina-Pirati\results_corrected_hp
